# Analysis — Lab 3 complexity audit

This notebook is the complexity analysis for the **analysis** module. Each nonempty source line appears below with an upper-bound or a note about its enclosing operation. The costs refer to the entire execution of that statement over enclosing loops; lines that form a multiline expression must be interpreted together. Detailed globally combined bounds are in `../../docs/complexity_summary.md`.

**Symbols.** `A` actors; `C=2` conditions; `R=C·A` recordings; `Z` ZIP entries; `N` samples; `K` 25 ms frame samples; `H` 10 ms hop samples; `W≈N/H` frames per recording; `W0` longer pYIN frames; `F=5` characteristics; `S=6` statistics; `d=4` classifier inputs; `M=2` models; `Pixels` number of saved image pixels; `T_pYIN` and `T_fit` are deliberately opaque library-level costs.

## Line-by-line analysis: `features.py`

| Line | Source statement | Estimated cost / reason |
|---:|---|---|
| 1 | `"""Acoustic feature extraction, using a reference and vectorized implementation."""` | Comment/docstring — no runtime work |
| 3 | `from __future__ import annotations` | O(1) declaration/binding; function call cost discussed below |
| 5 | `import librosa` | O(1) declaration/binding; function call cost discussed below |
| 6 | `import numpy as np` | O(1) declaration/binding; function call cost discussed below |
| 7 | `from scipy.signal.windows import hann` | O(1) declaration/binding; function call cost discussed below |
| 10 | `def window_features_vectorized(audio_matrix: np.ndarray, fs: int, window_ms: int = 25, hop_ms: int = 10, include_...` | O(1) declaration/binding; function call cost discussed below |
| 11 | `"""Compute 25 ms time/spectral features and framewise pYIN F0 for each actor.` | Comment/docstring — no runtime work |
| 13 | `Inputs: 2D matrix (actors, samples). Outputs: (actors, frames), with` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 14 | `F0 computed using a longer analysis window but the same hop length.` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 15 | `"""` | Comment/docstring — no runtime work |
| 16 | `if audio_matrix.ndim != 2:` | O(1) condition/control unless evaluating an included collection |
| 17 | `raise ValueError("Expected a 2-D array (actors, samples)")` | O(1) condition/control unless evaluating an included collection |
| 18 | `k = round(fs * window_ms / 1000)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 19 | `h = round(fs * hop_ms / 1000)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 20 | `if min(k, h) < 1 or audio_matrix.shape[1] < k:` | O(1) condition/control unless evaluating an included collection |
| 21 | `raise ValueError("Invalid window or recording length")` | O(1) condition/control unless evaluating an included collection |
| 23 | `# librosa framing along the last dimension gives (actors, K, W).` | Comment/docstring — no runtime work |
| 24 | `frames = librosa.util.frame(audio_matrix, frame_length=k, hop_length=h, axis=-1)` | O(1) view metadata; no sample copying |
| 25 | `frames = np.swapaxes(frames, -1, -2)  # (actors, W, K)` | O(1) view metadata; no sample copying |
| 26 | `rms = np.sqrt(np.mean(frames**2, axis=-1))` | O(R·W·K) vectorized reduction / comparison |
| 27 | `peak = np.max(np.abs(frames), axis=-1)` | O(R·W·K) vectorized reduction / comparison |
| 28 | `zcr = np.mean(np.diff(np.signbit(frames), axis=-1), axis=-1)` | O(R·W·K) vectorized reduction / comparison |
| 30 | `weighted = frames * hann(k, sym=False)` | O(R·W·K) multiply/memory |
| 31 | `spectrum = np.abs(np.fft.rfft(weighted, axis=-1))` | O(R·W·K log K) FFT of all short-time frames |
| 32 | `frequencies = np.fft.rfftfreq(k, d=1 / fs)` | O(K) frequency grid |
| 33 | `centroid = np.sum(spectrum * frequencies, axis=-1) / (np.sum(spectrum, axis=-1) + 1e-10)` | O(R·W·K) weighted spectral reduction |
| 35 | `result = {"RMS": rms, "Peak": peak, "ZCR": zcr, "Spectral Centroid": centroid}` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 36 | `if include_pitch:` | O(1) condition/control unless evaluating an included collection |
| 37 | `p = 2 ** int(np.ceil(np.log2(2 * fs / 65)))` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 38 | `if audio_matrix.shape[1] < p:` | O(1) condition/control unless evaluating an included collection |
| 39 | `raise ValueError("Recording too short for the minimum F0 (65 Hz)")` | O(1) condition/control unless evaluating an included collection |
| 40 | `f0, _, _ = librosa.pyin(audio_matrix, sr=fs, fmin=65, fmax=500,` | T_pYIN(R,N,P,H) — opaque library estimator |
| 41 | `frame_length=p, hop_length=h, center=False)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 42 | `result["F0"] = f0` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 43 | `return result` | O(1) to return references (ignoring any nested expression) |
| 46 | `def window_features_reference(audio_matrix: np.ndarray, fs: int, window_ms: int = 25, hop_ms: int = 10, include_p...` | O(1) declaration/binding; function call cost discussed below |
| 47 | `"""Loop-based baseline for four features; same pYIN estimator as optimized version.` | Comment/docstring — no runtime work |
| 49 | `Retaining identical mathematical definitions supports correctness checks.` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 50 | `"""` | Comment/docstring — no runtime work |
| 51 | `k = round(fs * window_ms / 1000)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 52 | `h = round(fs * hop_ms / 1000)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 53 | `n_frames = 1 + (audio_matrix.shape[1] - k) // h` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 54 | `if n_frames < 1:` | O(1) condition/control unless evaluating an included collection |
| 55 | `raise ValueError("Recording too short for framing")` | O(1) condition/control unless evaluating an included collection |
| 56 | `weights = hann(k, sym=False)` | O(K) to construct window coefficients |
| 57 | `frequencies = np.fft.rfftfreq(k, d=1 / fs)` | O(K) frequency grid |
| 58 | `values = {name: np.empty((len(audio_matrix), n_frames), dtype=float) for name in ("RMS", "Peak", "ZCR", "Spectral...` | O(R·W) allocation or O(R) for prediction arrays |
| 59 | `for actor, signal in enumerate(audio_matrix):` | O(R) loop headers (body total shown separately) |
| 60 | `for window in range(n_frames):` | O(R·W) nested loop iterations |
| 61 | `frame = signal[window*h:window*h+k]` | O(1) slice view, once per (actor, window) |
| 62 | `values["RMS"][actor, window] = np.sqrt(np.mean(frame**2))` | O(K) per (actor, window) — FFT O(K log K) if called |
| 63 | `values["Peak"][actor, window] = np.max(np.abs(frame))` | O(K) per (actor, window) — FFT O(K log K) if called |
| 64 | `values["ZCR"][actor, window] = np.mean(np.diff(np.signbit(frame)))` | O(K) per (actor, window) — FFT O(K log K) if called |
| 65 | `spectrum = np.abs(np.fft.rfft(frame * weights))` | O(R·W·K log K) FFT of all short-time frames |
| 66 | `values["Spectral Centroid"][actor, window] = (spectrum * frequencies).sum() / (spectrum.sum() + 1e-10)` | O(K) per (actor, window) — FFT O(K log K) if called |
| 67 | `if include_pitch:` | O(1) condition/control unless evaluating an included collection |
| 68 | `p = 2 ** int(np.ceil(np.log2(2 * fs / 65)))` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 69 | `values["F0"], _, _ = librosa.pyin(audio_matrix, sr=fs, fmin=65, fmax=500,` | T_pYIN(R,N,P,H) — opaque library estimator |
| 70 | `frame_length=p, hop_length=h, center=False)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 71 | `return values` | O(1) to return references (ignoring any nested expression) |

## Line-by-line analysis: `statistics.py`

| Line | Source statement | Estimated cost / reason |
|---:|---|---|
| 1 | `"""Statistical summary and paired tests for windowed acoustic measurements."""` | Comment/docstring — no runtime work |
| 3 | `from __future__ import annotations` | O(1) declaration/binding; function call cost discussed below |
| 5 | `import numpy as np` | O(1) declaration/binding; function call cost discussed below |
| 6 | `import pandas as pd` | O(1) declaration/binding; function call cost discussed below |
| 7 | `from scipy.stats import shapiro, wilcoxon` | O(1) declaration/binding; function call cost discussed below |
| 8 | `from statsmodels.stats.multitest import multipletests` | O(1) declaration/binding; function call cost discussed below |
| 10 | `STATISTICS = ("Mean", "Std", "Median", "Min", "Max", "Range")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 13 | `def summarize_reference(features: dict[str, np.ndarray], actors: list[str]) -> pd.DataFrame:` | O(1) declaration/binding; function call cost discussed below |
| 14 | `"""Original notebook-style stacked DataFrame + unstack/swaplevel summary."""` | Comment/docstring — no runtime work |
| 15 | `intermediate = pd.concat(` | O(1) structural / continuation |
| 16 | `[pd.DataFrame(values, index=actors) for values in features.values()],` | O(number of stored values) construction |
| 17 | `keys=features.keys(), names=["Characteristic", "Actor"]` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 18 | `)` | O(1) structural / continuation |
| 19 | `stats = pd.DataFrame({` | O(number of stored values) construction |
| 20 | `"Mean": intermediate.mean(axis=1),` | O(F·A·W) reduction; median up to O(F·A·W log W) worst-case |
| 21 | `"Std": intermediate.std(axis=1),` | O(F·A·W) reduction; median up to O(F·A·W log W) worst-case |
| 22 | `"Median": intermediate.median(axis=1),` | O(F·A·W) reduction; median up to O(F·A·W log W) worst-case |
| 23 | `"Min": intermediate.min(axis=1),` | O(F·A·W) reduction; median up to O(F·A·W log W) worst-case |
| 24 | `"Max": intermediate.max(axis=1),` | O(F·A·W) reduction; median up to O(F·A·W log W) worst-case |
| 25 | `"Range": intermediate.max(axis=1) - intermediate.min(axis=1),` | O(F·A·W) reduction; median up to O(F·A·W log W) worst-case |
| 26 | `})` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 27 | `stats = stats.unstack("Characteristic").swaplevel(0, 1, axis=1)` | O(F·A·S) output column/index arrangement (may allocate) |
| 28 | `stats = stats.reindex(columns=list(features), level=0)` | O(F·A·S) output column/index arrangement (may allocate) |
| 29 | `stats.columns.names = ("Characteristic", "Statistic")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 30 | `stats.index.name = "Actor"` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 31 | `return stats` | O(1) to return references (ignoring any nested expression) |
| 34 | `def summarize_optimized(features: dict[str, np.ndarray], actors: list[str]) -> pd.DataFrame:` | O(1) declaration/binding; function call cost discussed below |
| 35 | `"""Construct actor-row/feature-statistic-column MultiIndex without one giant frame table.` | Comment/docstring — no runtime work |
| 37 | `Pandas ignores undefined (unvoiced) pYIN F0 values by default.` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 38 | `"""` | Comment/docstring — no runtime work |
| 39 | `output = {}` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 40 | `for characteristic, values in features.items():` | O(F) iterations over characteristics |
| 41 | `small = pd.DataFrame(values, index=actors)` | O(A·W) for this feature; F repetitions |
| 42 | `minimum = small.min(axis=1)` | O(A·W) this feature; O(A·W log W) conservative for median |
| 43 | `maximum = small.max(axis=1)` | O(A·W) this feature; O(A·W log W) conservative for median |
| 44 | `output[(characteristic, "Mean")] = small.mean(axis=1)` | O(A·W) this feature; O(A·W log W) conservative for median |
| 45 | `output[(characteristic, "Std")] = small.std(axis=1)` | O(A·W) this feature; O(A·W log W) conservative for median |
| 46 | `output[(characteristic, "Median")] = small.median(axis=1)` | O(A·W) this feature; O(A·W log W) conservative for median |
| 47 | `output[(characteristic, "Min")] = minimum` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 48 | `output[(characteristic, "Max")] = maximum` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 49 | `output[(characteristic, "Range")] = maximum - minimum` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 50 | `summary = pd.DataFrame(output, index=actors).astype("float64")` | O(number of stored values) construction |
| 51 | `summary.columns.names = ("Characteristic", "Statistic")` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 52 | `summary.index.name = "Actor"` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 53 | `return summary` | O(1) to return references (ignoring any nested expression) |
| 56 | `def paired_tests(neutral: pd.DataFrame, sad: pd.DataFrame) -> pd.DataFrame:` | O(1) declaration/binding; function call cost discussed below |
| 57 | `"""Paired Shapiro–Wilk and Wilcoxon tests with Holm multiplicity correction.` | Comment/docstring — no runtime work |
| 59 | `Same actor must occur in both conditions. Test statistics are descriptive` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 60 | `and not used for selecting classifier features before cross-validation.` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 61 | `"""` | Comment/docstring — no runtime work |
| 62 | `if not neutral.index.equals(sad.index) or not neutral.columns.equals(sad.columns):` | O(1) condition/control unless evaluating an included collection |
| 63 | `raise ValueError("Paired tables require identical actor order and feature columns")` | O(1) condition/control unless evaluating an included collection |
| 64 | `differences = sad - neutral` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 65 | `rows = []` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 66 | `for feature, statistic in differences.columns:` | O(F·S) hypothesis tests |
| 67 | `delta = differences[(feature, statistic)].dropna()` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 68 | `if len(delta) >= 3 and delta.nunique() > 1:` | O(1) condition/control unless evaluating an included collection |
| 69 | `shapiro_p = float(shapiro(delta).pvalue)` | Statistical library cost (up to O(A log A) conservative/test) |
| 70 | `else:` | O(1) structural / continuation |
| 71 | `shapiro_p = np.nan` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 72 | `if delta.empty:` | O(1) condition/control unless evaluating an included collection |
| 73 | `wilcoxon_p = np.nan` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 74 | `elif (delta == 0).all():` | O(1) condition/control unless evaluating an included collection |
| 75 | `wilcoxon_p = 1.0` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 76 | `else:` | O(1) structural / continuation |
| 77 | `wilcoxon_p = float(wilcoxon(delta, alternative="two-sided").pvalue)` | Statistical library cost (up to O(A log A) conservative/test) |
| 78 | `rows.append((feature, statistic, len(delta), shapiro_p, wilcoxon_p))` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 79 | `result = pd.DataFrame(rows, columns=["Feature", "Statistic", "N pairs", "Shapiro p", "Wilcoxon p"])` | O(number of stored values) construction |
| 80 | `result = result.set_index(["Feature", "Statistic"])` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 81 | `valid = result["Wilcoxon p"].notna()` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 82 | `result["Holm-adjusted p"] = np.nan` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 83 | `if valid.any():` | O(1) condition/control unless evaluating an included collection |
| 84 | `result.loc[valid, "Holm-adjusted p"] = multipletests(result.loc[valid, "Wilcoxon p"], method="holm")[1]` | O(F·S log(F·S)) p-value sorting / correction |
| 85 | `return result` | O(1) to return references (ignoring any nested expression) |

## Line-by-line analysis: `classification.py`

| Line | Source statement | Estimated cost / reason |
|---:|---|---|
| 1 | `"""Actor-grouped cross-validation for neutral vs acted-sad speech."""` | Comment/docstring — no runtime work |
| 3 | `from __future__ import annotations` | O(1) declaration/binding; function call cost discussed below |
| 5 | `import numpy as np` | O(1) declaration/binding; function call cost discussed below |
| 6 | `import pandas as pd` | O(1) declaration/binding; function call cost discussed below |
| 7 | `from sklearn.base import clone` | O(1) declaration/binding; function call cost discussed below |
| 8 | `from sklearn.impute import SimpleImputer` | O(1) declaration/binding; function call cost discussed below |
| 9 | `from sklearn.linear_model import LogisticRegression` | O(1) declaration/binding; function call cost discussed below |
| 10 | `from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score` | O(1) declaration/binding; function call cost discussed below |
| 11 | `from sklearn.model_selection import LeaveOneGroupOut` | O(1) declaration/binding; function call cost discussed below |
| 12 | `from sklearn.pipeline import make_pipeline` | O(1) declaration/binding; function call cost discussed below |
| 13 | `from sklearn.preprocessing import StandardScaler` | O(1) declaration/binding; function call cost discussed below |
| 15 | `# Predeclared, domain-motivated columns. Never pick features using all labels before CV.` | Comment/docstring — no runtime work |
| 16 | `DEFAULT_COLUMNS = (("RMS", "Mean"), ("ZCR", "Mean"),` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 17 | `("Spectral Centroid", "Median"), ("F0", "Median"))` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 20 | `def grouped_classification(neutral: pd.DataFrame, sad: pd.DataFrame,` | O(1) declaration/binding; function call cost discussed below |
| 21 | `selected_columns=DEFAULT_COLUMNS) -> tuple[pd.DataFrame, pd.DataFrame]:` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 22 | `"""Cross-validate logistic regression, holding out both conditions of one actor.` | Comment/docstring — no runtime work |
| 24 | `Returns the aggregate metrics and one prediction per actor/condition.` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 25 | `"""` | Comment/docstring — no runtime work |
| 26 | `if len(neutral) < 3 or not neutral.index.equals(sad.index):` | O(1) condition/control unless evaluating an included collection |
| 27 | `raise ValueError("At least three matched actors in identical order are required")` | O(1) condition/control unless evaluating an included collection |
| 28 | `columns = list(selected_columns)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 29 | `X = pd.concat([neutral.loc[:, columns], sad.loc[:, columns]], ignore_index=True)` | O(F·A·W) materialization of stacked feature frame |
| 30 | `y = np.array([0] * len(neutral) + [1] * len(sad))` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 31 | `groups = np.tile(neutral.index.to_numpy(), 2)` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 32 | `models = {` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 33 | `"Logistic regression": make_pipeline(SimpleImputer(strategy="median"), LogisticRegression(max_iter=2000, solver="...` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 34 | `"Standardized logistic": make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max...` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 35 | `}` | O(1) structural / continuation |
| 36 | `metric_rows, prediction_rows = [], []` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 37 | `folds = LeaveOneGroupOut()` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 38 | `for name, model in models.items():` | O(M) model configurations |
| 39 | `probabilities = np.empty(len(y), dtype=float)` | O(R·W) allocation or O(R) for prediction arrays |
| 40 | `predicted = np.empty(len(y), dtype=int)` | O(R·W) allocation or O(R) for prediction arrays |
| 41 | `for train_idx, test_idx in folds.split(X, y, groups):` | O(A) actor-held-out folds per model |
| 42 | `fitted = clone(model).fit(X.iloc[train_idx], y[train_idx])` | T_fit(2A−2,d) per fold; M·A folds |
| 43 | `probabilities[test_idx] = fitted.predict_proba(X.iloc[test_idx])[:, 1]` | O(d) prediction per held-out recording, solver-dependent |
| 44 | `predicted[test_idx] = fitted.predict(X.iloc[test_idx])` | O(d) prediction per held-out recording, solver-dependent |
| 45 | `metric_rows.append({` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 46 | `"Model": name,` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 47 | `"Accuracy": accuracy_score(y, predicted),` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 48 | `"Precision": precision_score(y, predicted, zero_division=0),` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 49 | `"Recall": recall_score(y, predicted, zero_division=0),` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 50 | `"F1": f1_score(y, predicted, zero_division=0),` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 51 | `"ROC-AUC": roc_auc_score(y, probabilities),` | O(R log R) sorting scores per model |
| 52 | `})` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 53 | `prediction_rows.extend({"Model": name, "Actor": actor, "True": int(target), "Predicted": int(pred), "P(Sad)": prob}` | O(1) arithmetic/binding; see function-level bound for repeated execution |
| 54 | `for actor, target, pred, prob in zip(groups, y, predicted, probabilities))` | O(R) loop headers (body total shown separately) |
| 55 | `return pd.DataFrame(metric_rows).set_index("Model"), pd.DataFrame(prediction_rows)` | O(number of stored values) construction |

## Global algorithm bounds

| Algorithm | End-to-end bound |
|---|---|
| `window_features_reference` | O(R·W·K log K) + T_pYIN with O(R·W) Python loop iterations; O(R·W) outputs. |
| `window_features_vectorized` | O(R·W·K log K) + T_pYIN; far fewer Python loop operations, potentially large temporary O(R·W·K). |
| `summarize_reference` | O(F·R·W log W) conservative; temporary stacked table O(F·R·W). |
| `summarize_optimized` | O(F·R·W log W) conservative; one feature-sized intermediate O(R·max(W,W0)). |
| `paired_tests` | O(F·S·A log A) conservative + O(F·S log(F·S)) Holm correction. |
| `grouped_classification` | O(M·A·T_fit(2A−2,d)) plus inference / metric overhead; actor-held-out folds. |

**Optimization.** The loop baseline and the vectorized method compute the same values; tests assert numerical equivalence. Both have FFT-bound `O(R·W·K log K)` time, and both call the **same** pYIN routine. The per-feature summaries reduce intermediate memory from `O(F·R·W)` toward `O(R·W + F·R·S)` but can cost additional Pandas calls. The logistic model uses actor-level folds to avoid leakage.

In [ ]:
# Summary of the module: view the registered algorithm bounds without loading data.
complexities = [('window_features_reference', 'O(R·W·K log K) + T_pYIN with O(R·W) Python loop iterations; O(R·W) outputs.'), ('window_features_vectorized', 'O(R·W·K log K) + T_pYIN; far fewer Python loop operations, potentially large temporary O(R·W·K).'), ('summarize_reference', 'O(F·R·W log W) conservative; temporary stacked table O(F·R·W).'), ('summarize_optimized', 'O(F·R·W log W) conservative; one feature-sized intermediate O(R·max(W,W0)).'), ('paired_tests', 'O(F·S·A log A) conservative + O(F·S log(F·S)) Holm correction.'), ('grouped_classification', 'O(M·A·T_fit(2A−2,d)) plus inference / metric overhead; actor-held-out folds.')]
for name, complexity in complexities:
    print(f"{name}: {complexity}")